### Setting up

In [ ]:
# Add SSAS dlls to path
from sys import path
path.append('./Microsoft.AnalysisServices.Tabular')
path.append('./Microsoft.AnalysisServices.AdomdClient')

# Import packages
import pandas as pd
from os import environ
from pyadomd import Pyadomd
from auth import Auth

# Tenant/app settings
SERVER = 'server_connection_string' # Like 'powerbi://api.powerbi.com/v1.0/myorg/my_workspace'
MODEL_NAME = 'dataset_name'
TENANT_ID = environ.get('TENANT_ID', '')
CLIENT_ID = environ.get('CLIENT_ID', '')
CLIENT_SECRET = environ.get('CLIENT_SECRET', '')

In [ ]:
# Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = auth.get_token()

In [ ]:
connection_string_part1 = f'Provider=MSOLAP.8;Data Source={SERVER};Initial Catalog={MODEL_NAME};'
connection_string_part2 = f'User ID=;Password={token};Persist Security Info=True;Impersonation Level=Impersonate;'
connection_string = connection_string_part1 + connection_string_part2

results = {}

dax_queries = {
    'name': 'table_name',
    'query':
        '''
            EVALUATE table
        '''
}

In [ ]:
with Pyadomd(connection_string) as conn:
    with conn.cursor().execute(dax_queries['query']) as cursor:
        result = cursor.fetchall()
        results[dax_queries['name']] = {}
        results[dax_queries['name']]['columns'] = [c.name[c.name.find("[")+1:c.name.find("]")] for c in cursor.description]
        results[dax_queries['name']]['query_result'] = result

In [ ]:
columns = results['table_name']['columns']
data = results['table_name']['query_result']

df = pd.DataFrame(data, columns=columns, dtype=str)